Задание 1

Вам необходимо написать функцию, которая будет основана на поиске по сайту habr.com. Функция в качестве параметра должна принимать список запросов для поиска (например, ['python', 'анализ данных']) и на основе материалов, попавших в результаты поиска по каждому запросу, возвращать датафрейм вида:

<дата> - <заголовок> - <ссылка на материал>
В рамках задания предполагается работа только с одной (первой) страницей результатов поисковой выдачи для каждого запроса. Материалы в датафрейме не должны дублироваться, если они попадали в результаты поиска для нескольких запросов из списка.

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def search_habr(queries):
    articles = set()  # используем множество, чтобы избежать дубликатов
    base_url = 'https://habr.com/ru/search/?q='

    for query in queries:
        response = requests.get(base_url + query.replace(' ', '+'))
        soup = BeautifulSoup(response.text, 'html.parser')

        # находим блоки со статьями
        for item in soup.find_all('article'):
            title_tag = item.find('h2') # проверяем, существует ли заголовок статьи
            if title_tag:
                title = title_tag.text.strip()
                link_tag = title_tag.find('a') # проверяем наличие ссылки внутри заголовка
                if link_tag:
                    link = link_tag['href']
                    date_tag = item.find('time') # проверяем наличие элемента time для получения даты
                    if date_tag:
                        date = date_tag['title']
                        # добавляем в множество
                        articles.add((date, title, link))

    # преобразуем множество в датафрейм
    df = pd.DataFrame(list(articles), columns=['Дата', 'Заголовок', 'Ссылка'])

    # преобразуем колонку с датами в datetime и сортируем по убыванию
    df['Дата'] = pd.to_datetime(df['Дата'], errors='coerce')
    df = df.sort_values(by='Дата', ascending=False) 

    return df

# пример использования функции
queries = ['python', 'анализ данных']
result_df = search_habr(queries)
print(result_df)


                  Дата                                          Заголовок  \
3  2024-11-20 09:05:00     Как мы обновили курсы для Python-разработчиков   
36 2024-10-23 09:14:00  1С VS Python – новый выпуск ютуб-шоу «Согласен...   
34 2024-09-17 16:00:00  ИТМО провёл исследование open source в сферах ...   
25 2024-09-02 10:27:00  55% Python-разработчиков используют Linux-окру...   
35 2024-07-11 09:57:00  Инсайдерам Microsoft 365 станет доступен редак...   
15 2024-07-11 07:49:00  Релиз утилиты CatLock 1.0.0 для Windows, предн...   
4  2024-07-01 14:46:00  Релиз открытой библиотеки для быстрой обработк...   
2  2024-04-27 10:20:00  Начните учиться бесплатно на курсе «Python для...   
31 2023-11-09 14:05:00  Alchemer совместно с JetBrains и Python Softwa...   
12 2023-10-20 07:17:00  Автор курсов по Python и Pandas жалуется на ве...   
9  2023-10-09 15:23:00  Три уровня погружения в Python. Запись докладо...   
30 2023-10-09 08:39:00  JetBrains и Python Software Foundation рассказ...   

Задание 2

Функция из первой части задания должна быть расширена следующим образом:

кроме списка ключевых слов для поиска необходимо объявить параметр с количеством страниц поисковой выдачи. Т.е. при передаче в функцию аргумента 4 необходимо получить материалы с первых 4 страниц результатов;
в датафрейме должны быть столбцы с полным текстом найденных материалов и количеством лайков:
<дата> - <заголовок> - <ссылка на материал> - <текст материала> - <количество лайков>

In [15]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def search_habr(queries, pages=1): # добавлен параметр pages для указания количества страниц, которые нужно обработать
    articles = set()  # используем множество, чтобы избежать дубликатов
    base_url = 'https://habr.com'
    search_url = f'{base_url}/ru/search/?q=' # добавим условный оператор, который будет проверять, начинается ли ссылка с "http"; если нет, то добавляется базовый URL к относительной ссылке 

    for query in queries:
        for page in range(pages):
            response = requests.get(search_url + query.replace(' ', '+'))
            soup = BeautifulSoup(response.text, 'html.parser')

        # находим блоки со статьями
        for item in soup.find_all('article'):
            title_tag = item.find('h2') # проверяем, существует ли заголовок статьи
            if title_tag:
                title = title_tag.text.strip()
                link_tag = title_tag.find('a') # проверяем наличие ссылки внутри заголовка
                if link_tag:
                    link = link_tag['href']
                    if link.startswith('/'):
                        link = base_url + link
                    date_tag = item.find('time') # проверяем наличие элемента time для получения даты
                    if date_tag:
                        date = date_tag['title']
                        # получаем количество лайков
                        like_tag = item.find('span', class_='voting-wjt__count')
                        likes = like_tag.text.strip() if like_tag else '0'
                        # получаем полный текст материала
                        article_response = requests.get(link)
                        article_soup = BeautifulSoup(article_response.text, 'html.parser')
                        content_tag = article_soup.find('div', class_='post__body-post')
                        full_text = content_tag.text.strip() if content_tag else 'Текст недоступен'
                        # добавляем в множество
                        articles.add((date, title, link, full_text, likes))

    # преобразуем множество в датафрейм
    df = pd.DataFrame(list(articles), columns=['Дата', 'Заголовок', 'Ссылка', 'Текст', 'Количество лайков'])

    # преобразуем колонку с датами в datetime и сортируем по убыванию
    df['Дата'] = pd.to_datetime(df['Дата'], errors='coerce')
    df = df.sort_values(by='Дата', ascending=False) 

    return df

# пример использования функции
queries = ['python', 'анализ данных']
result_df = search_habr(queries, pages=4)
print(result_df)

                  Дата                                          Заголовок  \
38 2024-11-20 09:05:00     Как мы обновили курсы для Python-разработчиков   
26 2024-10-23 09:14:00  1С VS Python – новый выпуск ютуб-шоу «Согласен...   
15 2024-09-17 16:00:00  ИТМО провёл исследование open source в сферах ...   
8  2024-09-02 10:27:00  55% Python-разработчиков используют Linux-окру...   
27 2024-07-11 09:57:00  Инсайдерам Microsoft 365 станет доступен редак...   
30 2024-07-11 07:49:00  Релиз утилиты CatLock 1.0.0 для Windows, предн...   
3  2024-07-01 14:46:00  Релиз открытой библиотеки для быстрой обработк...   
20 2024-04-27 10:20:00  Начните учиться бесплатно на курсе «Python для...   
28 2023-11-09 14:05:00  Alchemer совместно с JetBrains и Python Softwa...   
39 2023-10-20 07:17:00  Автор курсов по Python и Pandas жалуется на ве...   
22 2023-10-09 15:23:00  Три уровня погружения в Python. Запись докладо...   
12 2023-10-09 08:39:00  JetBrains и Python Software Foundation рассказ...   